# PC-FNO3D réel (Grand Tunis) v6 — affinage visant DSO1 et DSO3 (Colab, A100)

La v4 livrée manque **DSO1 au seul niveau 25 m** (RMSE 0,528 pour 0,5) et **DSO3** (rugosité 0,805
pour [0,9 ; 1,1]) : le champ est trop lisse. Étapes : (1) poids u, v par niveau mesurés sur la
**validation** de la v4 ; (2) affinage 15 époques depuis la v4, avec un **terme de rugosité par
niveau** (`loss.lambda_rough`), l'époque étant choisie sur DSO1, DSO2 et DSO3 de la **validation** ;
(3) test 2022 (w par continuité), lu une fois ; (4) vraies mesures ; (5) **décision**
(`scripts/decision_v6_grand_tunis.py`, règle fixée le 07/11/2026 avant le test) : la v6 n'est livrée
que si DSO1 et DSO3 sont conformes, sans recul sur DSO2, DSO4, DSO5 et les vraies mesures ; sinon la
v4 reste ; (6) DSO6 si la v6 est livrée. Aucun bruit n'est ajouté aux prédictions.
`decision_v4.json` (lu par le service) n'est pas modifié. La session s'arrête à la fin.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

## 2. Cache (3 parties) et observations

In [ ]:
# cache : les 3 parties du Drive -> disque local ; index fusionne
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (GT / f"cache_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())

# la grille complete (aneo_grid_points.csv) est dans jeu.zip
if not (GT / "best_dso_v7.pt").exists():
    raise FileNotFoundError("deposer best_dso_v7.pt (poids v7) dans MyDrive/pcfno3d_grand_tunis/")
lancer("controle", "python -u -m pcfno3d -c grand_tunis_v6_dso_colab check")

# observations reelles du domaine (affinage et jugement)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print(sorted(p.name for p in OBS.iterdir()))

## 3. Mémoire du GPU

In [ ]:
# memoire : un pas complet (avant + arriere) sur la grille entiere ; recalcul des activations
# couche par couche si le GPU ne suffit pas (memes resultats, ~30 % plus lent)
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/grand_tunis_v6_dso_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Poids, affinage v6, test, vraies mesures, décision, fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

from google.colab import runtime
import shlex
import traceback
RUNS = GT / "runs"
V4, V6 = "grand_tunis_v4_dso_colab", "grand_tunis_v6_dso_colab"
R4, R6 = RUNS / "grand_tunis_v4_dso", RUNS / "grand_tunis_v6_dso"
R6.mkdir(parents=True, exist_ok=True)
POIDS = R6 / "uv_level_weight.json"
INIT = shlex.quote(str(choisir_poids(R4)))

def entraine():
    h = R6 / "history.json"
    return h.exists() and any(r.get("epoch") == 14 for r in json.loads(h.read_text()))

def evalue(run, dossier):
    f = run / "eval_pcfno3d" / dossier / "test_par_niveau_synoptique.csv"
    return f.exists() and "rugosite" in f.read_text().splitlines()[0]

etapes = (
    # poids u, v par niveau : RMSE de VALIDATION de la v4 (jamais le test)
    ("poids_v6", f"python -u -m pcfno3d -c {V4} {OPTS} --set inference.w_continuite=true dso-weights --checkpoint {INIT} --out {POIDS}",
     POIDS.exists),
    ("entrainer_v6", f"python -u -m pcfno3d -c {V6} {OPTS} --set loss.uv_level_weight={POIDS} "
                     f"--set train.init_from={INIT} train", entraine),
    ("evaluer_v6_wcont", f"python -u -m pcfno3d -c {V6} {OPTS} evaluate", lambda: evalue(R6, "best_wcont")),
    ("valider_v6", f"python -u scripts/valider_grand_tunis_reel.py -c {V6} --obs {OBS}",
     lambda: (R6 / "validation_reelle" / "test" / "stations_bilan.csv").exists()),
    # variante PROPOSEE (DSO3 sur la partie previsible) : publiee, jamais lue par la decision
    ("variantes_v6", f"python -u scripts/dso_variantes_grand_tunis.py -c {V6} --donnees {DON / 'grand_tunis'} --w-continuite",
     lambda: (R6 / "eval_pcfno3d" / "best_wcont" / "dso3_previsible.json").exists()),
)
try:
    for nom, cmd, fait in etapes:
        if fait():
            print(f"[{nom}] deja fait : saute")
            continue
        try:
            lancer(nom, cmd)
        except Exception:
            if nom in ("poids_v6", "entrainer_v6", "evaluer_v6_wcont"):
                raise                                  # sans eux, pas de decision possible
            traceback.print_exc()
    DEC = "python -u scripts/decision_v6_grand_tunis.py --runs " + str(RUNS)
    code_dec = subprocess.run(DEC, shell=True, cwd="/content/PFE").returncode
    if code_dec not in (0, 3):
        raise RuntimeError(f"decision v6 impossible (code {code_dec}) : evaluations incompletes")
    if code_dec == 0:
        # v6 livree : DSO6 sur la v6 sans reentrainement (calibration validation, verdict test)
        CAL = R6 / "fiabilite_pcfno3d" / "best_wcont" / "calibration_validation.json"
        D6 = R6 / "eval_pcfno3d" / "best_wcont" / "dso6_modele_livre.json"
        if not CAL.exists():
            lancer("calibrer_dso6_v6", f"python -u -m pcfno3d -c {V6} {OPTS} calibrate-reliability --out {CAL}")
        if not D6.exists():
            try:
                lancer("dso6_v6", f"python -u -m pcfno3d -c {V6} {OPTS} dso6 --calibration {CAL} --out {D6}")
            except RuntimeError as exc:          # code 3 = NON CONFORME : le verdict est ecrit
                print(exc)
    print((RUNS / "rapport_v6.md").read_text(encoding="utf-8"))
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()